In [5]:
# Cell 1: Imports, config, load model
import os
import sys

# Move to project root so relative paths in .env work
os.chdir("F:/ML PROJECT/LedgerWatch-AI/LedgerWatch-AI")
sys.path.insert(0, os.getcwd())  # Ensure src/ is on path

import numpy as np
import pandas as pd
import joblib
import shap
import matplotlib.pyplot as plt

from src.config import settings
from src.explain import (
    load_model_and_features,
    compute_shap_values,
    explain_transaction,
    get_global_feature_importance,
    plot_waterfall,
    plot_summary_beeswarm,
    plot_feature_importance_bar,
    export_shap_summary,
)

print(f"Python: {sys.version}")
print(f"SHAP version: {shap.__version__}")
print(f"Working directory: {os.getcwd()}")
print(f"Model path: {settings.MODEL_PATH}")
print(f"Features path: {settings.PROCESSED_DATA_PATH}")

# Load model
model, feature_names = load_model_and_features()
print(f"\nModel loaded: {type(model).__name__}")
print(f"Feature count: {len(feature_names)}")
print(f"First 5 features: {feature_names[:5]}")

Python: 3.10.20 | packaged by Anaconda, Inc. | (main, Mar 11 2026, 17:42:35) [MSC v.1942 64 bit (AMD64)]
SHAP version: 0.44.0
Working directory: F:\ML PROJECT\LedgerWatch-AI\LedgerWatch-AI
Model path: saved_models/isolation_forest_v1.0.0.joblib
Features path: data/processed/features.csv

Model loaded: IsolationForest
Feature count: 24
First 5 features: ['amount_log', 'is_round_amount', 'amount_to_balance_ratio', 'balance_diff_orig', 'balance_diff_dest']


In [6]:
# Cell 2: Load stratified sample and compute SHAP values
import pandas as pd

# Load full features
df = pd.read_csv(settings.PROCESSED_DATA_PATH)
print(f"Full dataset: {len(df):,} rows, {len(df.columns)} columns")

# Stratified sample for SHAP (5K rows = fast but representative)
SAMPLE_SIZE = 5000

fraud_df = df[df["isFraud"] == 1]
normal_df = df[df["isFraud"] == 0]

n_fraud = min(len(fraud_df), max(int(SAMPLE_SIZE * 0.15), 50))
n_normal = SAMPLE_SIZE - n_fraud

fraud_sample = fraud_df.sample(n=n_fraud, random_state=42)
normal_sample = normal_df.sample(n=n_normal, random_state=42)

df_sample = pd.concat([fraud_sample, normal_sample]).sample(frac=1, random_state=42).reset_index(drop=True)

X = df_sample[feature_names].reset_index(drop=True)
y = df_sample["isFraud"].reset_index(drop=True)

print(f"\nSample: {len(X):,} rows")
print(f"Fraud count: {y.sum():,} ({y.mean()*100:.2f}%)")
print(f"Normal count: {(y==0).sum():,}")

# Compute SHAP values (~30-60 seconds for 5K rows)
print("\nComputing SHAP values... (this may take 30-60s)")
shap_values = compute_shap_values(model, X, feature_names)
print(f"SHAP values shape: {shap_values.shape}")
print(f"SHAP mean: {shap_values.mean():.6f}")
print(f"SHAP std:  {shap_values.std():.6f}")

Full dataset: 6,362,620 rows, 35 columns

Sample: 5,000 rows
Fraud count: 750 (15.00%)
Normal count: 4,250

Computing SHAP values... (this may take 30-60s)
SHAP values shape: (5000, 24)
SHAP mean: 0.023055
SHAP std:  0.236890


In [7]:
# Cell 3: Global feature importance
importance_df = get_global_feature_importance(shap_values, feature_names)

print("=" * 60)
print("GLOBAL FEATURE IMPORTANCE (Mean |SHAP|)")
print("=" * 60)

for i, row in importance_df.head(10).iterrows():
    bar = "█" * int(row["mean_abs_shap_pct"] / 2)
    print(f"{i+1:2d}. {row['feature']:25s} | {row['mean_abs_shap']:.6f} | {row['mean_abs_shap_pct']:5.1f}% {bar}")

print(f"\nTop 5 features account for {importance_df.head(5)['mean_abs_shap_pct'].sum():.1f}% of total importance")

# Save bar plot
path = plot_feature_importance_bar(importance_df, output_path="docs/day8_shap_importance_bar.png")
print(f"\nPlot saved: {path}")

GLOBAL FEATURE IMPORTANCE (Mean |SHAP|)
 1. type_TRANSFER             | 0.269373 |   8.9% ████
 2. balance_diff_dest         | 0.257322 |   8.5% ████
 3. balance_change_orig       | 0.251170 |   8.3% ████
 4. is_new_dest               | 0.224248 |   7.4% ███
 5. zero_balance_dest         | 0.201508 |   6.6% ███
 6. amount_to_balance_ratio   | 0.175686 |   5.8% ██
 7. hour_of_step_sin          | 0.165640 |   5.5% ██
 8. type_CASH_IN              | 0.156914 |   5.2% ██
 9. hour_of_step_cos          | 0.154408 |   5.1% ██
10. hour_of_step              | 0.133033 |   4.4% ██

Top 5 features account for 39.7% of total importance

Plot saved: docs/day8_shap_importance_bar.png


In [8]:
# Cell 4: Pick sample transactions for waterfall plots
import numpy as np

# Compute risk scores (higher = more anomalous)
raw_scores = model.decision_function(X.values)
risk_scores = -raw_scores  # flip: higher = more anomalous

# Find indices
fraud_idx = int(np.argsort(risk_scores)[-1])   # highest risk
normal_idx = int(np.argsort(risk_scores)[0])   # lowest risk
mid_idx = int(np.argsort(risk_scores)[len(risk_scores) // 2])  # median risk

print(f"Fraud-like sample  (idx={fraud_idx}): risk_score={risk_scores[fraud_idx]:.4f}, actual_label={y.iloc[fraud_idx]}")
print(f"Normal sample      (idx={normal_idx}): risk_score={risk_scores[normal_idx]:.4f}, actual_label={y.iloc[normal_idx]}")
print(f"Medium-risk sample (idx={mid_idx}): risk_score={risk_scores[mid_idx]:.4f}, actual_label={y.iloc[mid_idx]}")

# Generate explanations
exp_fraud = explain_transaction(model, X.iloc[fraud_idx], feature_names)
exp_normal = explain_transaction(model, X.iloc[normal_idx], feature_names)
exp_mid = explain_transaction(model, X.iloc[mid_idx], feature_names)

print(f"\nFraud-like top 3 drivers:")
for c in exp_fraud["contributions"][:3]:
    direction = "↑ anomaly" if c["shap_value"] > 0 else "↓ normal"
    print(f"  {c['feature']:25s} = {c['value']:10.3f} | SHAP = {c['shap_value']:+.4f} ({direction})")

print(f"\nNormal top 3 drivers:")
for c in exp_normal["contributions"][:3]:
    direction = "↑ anomaly" if c["shap_value"] > 0 else "↓ normal"
    print(f"  {c['feature']:25s} = {c['value']:10.3f} | SHAP = {c['shap_value']:+.4f} ({direction})")

X does not have valid feature names, but IsolationForest was fitted with feature names
[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done 199 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done 200 out of 200 | elapsed:    0.0s finished


Fraud-like sample  (idx=1148): risk_score=0.1135, actual_label=0
Normal sample      (idx=1772): risk_score=-0.2156, actual_label=0
Medium-risk sample (idx=2026): risk_score=-0.1511, actual_label=0


X does not have valid feature names, but IsolationForest was fitted with feature names
[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done 199 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done 200 out of 200 | elapsed:    0.0s finished
X does not have valid feature names, but IsolationForest was fitted with feature names
[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done 199 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done 200 out of 200 | elapsed:    0.0s finished



Fraud-like top 3 drivers:
  balance_diff_dest         = 7125721.630 | SHAP = +1.3574 (↑ anomaly)
  balance_diff_orig         = 9933976.000 | SHAP = +1.2951 (↑ anomaly)
  is_round_amount           =      1.000 | SHAP = +1.2276 (↑ anomaly)

Normal top 3 drivers:
  is_new_dest               =      0.000 | SHAP = -0.2794 (↓ normal)
  zero_balance_orig         =      1.000 | SHAP = -0.2084 (↓ normal)
  zero_balance_dest         =      0.000 | SHAP = -0.1809 (↓ normal)


X does not have valid feature names, but IsolationForest was fitted with feature names
[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done 199 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done 200 out of 200 | elapsed:    0.0s finished


In [9]:
# Cell 5: Fix sample selection — explicitly pick fraud + normal + medium
import numpy as np

# Compute risk scores properly (pass DataFrame, not .values)
raw_scores = model.decision_function(X)  # DataFrame keeps feature names
risk_scores = -raw_scores

# Explicitly pick: highest-risk fraud, lowest-risk normal, median overall
fraud_mask = y == 1
normal_mask = y == 0

fraud_idx = int(np.argmax(risk_scores[fraud_mask]))  # highest risk among frauds
fraud_idx = fraud_mask[fraud_mask].index[fraud_idx]   # map back to full index

normal_idx = int(np.argmin(risk_scores[normal_mask]))  # lowest risk among normals
normal_idx = normal_mask[normal_mask].index[normal_idx]

mid_idx = int(np.argsort(risk_scores)[len(risk_scores) // 2])

print(f"Fraud sample       (idx={fraud_idx}): risk_score={risk_scores[fraud_idx]:.4f}, actual_label={y.iloc[fraud_idx]}")
print(f"Normal sample      (idx={normal_idx}): risk_score={risk_scores[normal_idx]:.4f}, actual_label={y.iloc[normal_idx]}")
print(f"Medium-risk sample (idx={mid_idx}): risk_score={risk_scores[mid_idx]:.4f}, actual_label={y.iloc[mid_idx]}")

# Generate explanations
exp_fraud = explain_transaction(model, X.iloc[fraud_idx], feature_names)
exp_normal = explain_transaction(model, X.iloc[normal_idx], feature_names)
exp_mid = explain_transaction(model, X.iloc[mid_idx], feature_names)

print(f"\n{'='*60}")
print("FRAUD SAMPLE — Top 5 drivers:")
print(f"{'='*60}")
for c in exp_fraud["contributions"][:5]:
    direction = "↑ anomaly" if c["shap_value"] > 0 else "↓ normal"
    print(f"  {c['feature']:25s} = {c['value']:10.3f} | SHAP = {c['shap_value']:+.4f} ({direction})")

print(f"\n{'='*60}")
print("NORMAL SAMPLE — Top 5 drivers:")
print(f"{'='*60}")
for c in exp_normal["contributions"][:5]:
    direction = "↑ anomaly" if c["shap_value"] > 0 else "↓ normal"
    print(f"  {c['feature']:25s} = {c['value']:10.3f} | SHAP = {c['shap_value']:+.4f} ({direction})")

[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done 199 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done 200 out of 200 | elapsed:    0.0s finished


Fraud sample       (idx=3194): risk_score=0.1063, actual_label=1
Normal sample      (idx=1772): risk_score=-0.2156, actual_label=0
Medium-risk sample (idx=2026): risk_score=-0.1511, actual_label=0


X does not have valid feature names, but IsolationForest was fitted with feature names
[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done 199 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done 200 out of 200 | elapsed:    0.0s finished
X does not have valid feature names, but IsolationForest was fitted with feature names
[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done 199 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done 200 out of 200 | elapsed:    0.0s finished



FRAUD SAMPLE — Top 5 drivers:
  balance_diff_dest         = 10000000.000 | SHAP = +1.4564 (↑ anomaly)
  is_round_amount           =      1.000 | SHAP = +1.2680 (↑ anomaly)
  type_TRANSFER             =      1.000 | SHAP = +0.9738 (↑ anomaly)
  balance_change_orig       = -10000000.000 | SHAP = +0.8224 (↑ anomaly)
  hour_of_step_cos          =      0.966 | SHAP = +0.6754 (↑ anomaly)

NORMAL SAMPLE — Top 5 drivers:
  is_new_dest               =      0.000 | SHAP = -0.2794 (↓ normal)
  zero_balance_orig         =      1.000 | SHAP = -0.2084 (↓ normal)
  zero_balance_dest         =      0.000 | SHAP = -0.1809 (↓ normal)
  hour_of_step_sin          =     -0.966 | SHAP = -0.1594 (↓ normal)
  type_CASH_IN              =      0.000 | SHAP = -0.1378 (↓ normal)


X does not have valid feature names, but IsolationForest was fitted with feature names
[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done 199 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done 200 out of 200 | elapsed:    0.0s finished


In [10]:
# Cell 6: Generate waterfall plots
print("Generating waterfall plots...")

path_fraud = plot_waterfall(
    exp_fraud, feature_names,
    output_path="docs/day8_shap_waterfall_fraud.png",
    title="SHAP Waterfall — High-Risk (Fraud) Transaction",
    max_display=10,
)
print(f"Fraud waterfall:  {path_fraud}")

path_normal = plot_waterfall(
    exp_normal, feature_names,
    output_path="docs/day8_shap_waterfall_normal.png",
    title="SHAP Waterfall — Low-Risk (Normal) Transaction",
    max_display=10,
)
print(f"Normal waterfall: {path_normal}")

path_mid = plot_waterfall(
    exp_mid, feature_names,
    output_path="docs/day8_shap_waterfall_mid.png",
    title="SHAP Waterfall — Medium-Risk Transaction",
    max_display=10,
)
print(f"Mid waterfall:    {path_mid}")

Generating waterfall plots...
Fraud waterfall:  docs/day8_shap_waterfall_fraud.png
Normal waterfall: docs/day8_shap_waterfall_normal.png
Mid waterfall:    docs/day8_shap_waterfall_mid.png


In [11]:
# Cell 7: SHAP summary beeswarm plot
print("Generating SHAP beeswarm summary plot...")

path_summary = plot_summary_beeswarm(
    shap_values, X, feature_names,
    output_path="docs/day8_shap_summary.png",
    max_display=15,
)
print(f"Summary plot: {path_summary}")

# Also show a quick text summary of the plot
print(f"\n{'='*60}")
print("PLOT SUMMARY")
print(f"{'='*60}")
print(f"Beeswarm plot saved: docs/day8_shap_summary.png")
print(f"Shows all 5,000 samples × top 15 features")
print(f"Red = high feature value pushes toward ANOMALY")
print(f"Blue = low feature value pushes toward NORMAL")

Generating SHAP beeswarm summary plot...
Summary plot: docs/day8_shap_summary.png

PLOT SUMMARY
Beeswarm plot saved: docs/day8_shap_summary.png
Shows all 5,000 samples × top 15 features
Red = high feature value pushes toward ANOMALY
Blue = low feature value pushes toward NORMAL


In [12]:
# Cell 8: Export JSON summary + final metrics
print("Exporting SHAP summary JSON...")

json_path = export_shap_summary(
    importance_df,
    [exp_fraud, exp_normal, exp_mid],
    output_path="docs/day8_shap_summary.json",
)
print(f"JSON saved: {json_path}")

# Print final Day 8 summary
print(f"\n{'='*60}")
print("DAY 8: SHAP EXPLAINABILITY — SUMMARY")
print(f"{'='*60}")
print(f"Sample size:           {len(X):,} rows")
print(f"Fraud in sample:       {y.sum():,} ({y.mean()*100:.1f}%)")
print(f"SHAP values shape:     {shap_values.shape}")
print(f"Top feature:           {importance_df.iloc[0]['feature']}")
print(f"Top feature importance: {importance_df.iloc[0]['mean_abs_shap']:.4f} ({importance_df.iloc[0]['mean_abs_shap_pct']:.1f}%)")
print(f"\nFiles generated:")
print(f"  docs/day8_shap_importance_bar.png")
print(f"  docs/day8_shap_summary.png")
print(f"  docs/day8_shap_waterfall_fraud.png")
print(f"  docs/day8_shap_waterfall_normal.png")
print(f"  docs/day8_shap_waterfall_mid.png")
print(f"  docs/day8_shap_summary.json")
print(f"\nDay 8 complete. Ready for documentation update.")

Exporting SHAP summary JSON...
JSON saved: docs/day8_shap_summary.json

DAY 8: SHAP EXPLAINABILITY — SUMMARY
Sample size:           5,000 rows
Fraud in sample:       750 (15.0%)
SHAP values shape:     (5000, 24)
Top feature:           type_TRANSFER
Top feature importance: 0.2694 (8.9%)

Files generated:
  docs/day8_shap_importance_bar.png
  docs/day8_shap_summary.png
  docs/day8_shap_waterfall_fraud.png
  docs/day8_shap_waterfall_normal.png
  docs/day8_shap_waterfall_mid.png
  docs/day8_shap_summary.json

Day 8 complete. Ready for documentation update.
